# ANFIS Data Preparation

This notebook prepares the dataset for the Neuro-Fuzzy system by defining the input features (Archetypes, Deltas) and generating the target supervisory signal (Adaptation Multiplier).

In [ ]:
import pandas as pd
import os
import numpy as np

INPUT_PATH = 'data/processed/5_clustered_telemetry.csv'
OUTPUT_PATH = 'data/processed/6_anfis_dataset.csv'

print("Loading clustered telemetry data...")
if os.path.exists(INPUT_PATH):
    df = pd.read_csv(INPUT_PATH)
    print(f"Loaded {len(df)} rows.")
    
    # Select Inputs for ANFIS
    # 1. State: Archetype Percentages (Combat, Collect, Explore)
    # 2. Change: Deltas (Delta_Combat, Delta_Collect, Delta_Explore)
    input_features = ['pct_combat', 'pct_collect', 'pct_explore', 
                      'delta_combat', 'delta_collect', 'delta_explore']
                      
    # Target Generation (Heuristic)
    # Logic: 
    # Base Multiplier = 1.0
    # Penalty: 0.1 per death (Makes game easier)
    # Reward: 0.05 * Normalized Total Activity Score (Reward engagement with difficulty)
    # This implies: If you are active and not dying, game gets harder.
    # If you are dying, game gets easier.
    
    # Note: 'score_total' in previous steps was sum of 0-1 features. 
    # It's likely in range 0-3 depending on how many features.
    # For safety, let's re-normalize 'activity_intensity' to 0-1 range relative to pop max.
    
    if 'score_total' in df.columns:
        max_score = df['score_total'].max()
        if max_score > 0:
            df['activity_intensity'] = df['score_total'] / max_score
        else:
            df['activity_intensity'] = 0.0
    else:
        # Fallback if score_total wasn't saved (it should have been in step 4)
        # Recompute basic intensity from pct columns? No, pct sums to 1.
        # Assuming score_total is present from step 4 output.
        print("Warning: score_total not found. Using default intensity 0.5")
        df['activity_intensity'] = 0.5

    # Apply Heuristic
    df['target_multiplier'] = 1.0 - (0.1 * df['death_count'].fillna(0)) + (0.05 * df['activity_intensity'])
    
    # Clip to [0.5, 1.5] for stability
    df['target_multiplier'] = df['target_multiplier'].clip(0.5, 1.5)
    
    # Save Dataset
    final_cols = ['userId', 'timestamp', 'cluster'] + input_features + ['target_multiplier']
    anfis_df = df[final_cols].copy()
    
    anfis_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nSaved ANFIS dataset to {OUTPUT_PATH}")
    print("\n--- Target Stats ---")
    print(anfis_df['target_multiplier'].describe())
else:
    print(f"Error: {INPUT_PATH} not found.")